In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16

In [3]:
data_dir = r"D:\SmartVision\DataSets\smartvision_dataset\classification"

img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/train",
    image_size=img_size,
    batch_size=batch_size
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/val",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/test",
    image_size=img_size,
    batch_size=batch_size
)

Found 1820 files belonging to 26 classes.
Found 390 files belonging to 26 classes.
Found 390 files belonging to 26 classes.


In [4]:
class_names = train_ds.class_names
print("Classes:", class_names)

Classes: ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [5]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [6]:
IMG_SIZE = (224, 224)
NUM_CLASSES = len(class_names)   # from earlier step

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


In [7]:
for layer in base_model.layers:
    layer.trainable = False

In [8]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,89

 Total params: 14,883,290 (56.78 MB)

 Trainable params: 168,090 (656.60 KB)

 Non-trainable params: 14,715,200 (56.13 MB)

In [11]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "vgg16_best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        verbose=1
    )
]

In [12]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.0368 - loss: 3.8343
Epoch 1: val_accuracy improved from None to 0.03590, saving model to vgg16_best_model.h5



Epoch 1: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 695s 12s/step - accuracy: 0.0407 - loss: 3.7771 - val_accuracy: 0.0359 - val_loss: 3.2590 - learning_rate: 1.0000e-04
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.0672 - loss: 3.5833
Epoch 2: val_accuracy improved from 0.03590 to 0.04615, saving model to vgg16_best_model.h5



Epoch 2: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 672s 12s/step - accuracy: 0.0676 - loss: 3.5444 - val_accuracy: 0.0462 - val_loss: 3.2165 - learning_rate: 1.0000e-04
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.0740 - loss: 3.3898
Epoch 3: val_accuracy improved from 0.04615 to 0.09487, saving model to vgg16_best_model.h5



Epoch 3: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 679s 12s/step - accuracy: 0.0819 - loss: 3.3328 - val_accuracy: 0.0949 - val_loss: 3.1748 - learning_rate: 1.0000e-04
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.1164 - loss: 3.1879
Epoch 4: val_accuracy improved from 0.09487 to 0.16410, saving model to vgg16_best_model.h5



Epoch 4: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 672s 12s/step - accuracy: 0.1214 - loss: 3.1434 - val_accuracy: 0.1641 - val_loss: 3.1286 - learning_rate: 1.0000e-04
Epoch 5/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.1514 - loss: 3.0454
Epoch 5: val_accuracy improved from 0.16410 to 0.17179, saving model to vgg16_best_model.h5


...


Epoch 6: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 668s 12s/step - accuracy: 0.1753 - loss: 2.9175 - val_accuracy: 0.2231 - val_loss: 2.9916 - learning_rate: 1.0000e-04
Epoch 7/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.1959 - loss: 2.8335
Epoch 7: val_accuracy did not improve from 0.22308
57/57 ━━━━━━━━━━━━━━━━━━━━ 705s 12s/step - accuracy: 0.2005 - loss: 2.8355 - val_accuracy: 0.1949 - val_loss: 2.9151 - learning_rate: 1.0000e-04
Epoch 8/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.2285 - loss: 2.7329
Epoch 8: val_accuracy did not improve from 0.22308
57/57 ━━━━━━━━━━━━━━━━━━━━ 667s 12s/step - accuracy: 0.2297 - loss: 2.7379 - val_accuracy: 0.2154 - val_loss: 2.8340 - learning_rate: 1.0000e-04
Epoch 9/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.2458 - loss: 2.6292
Epoch 9: val_accuracy did not improve from 0.22308
57/57 ━━━━━━━━━━━━━━━━━━━━ 705s 12s/step - accuracy: 0.2390 - loss: 2.6581 - val_accuracy: 0.1949 - val_l


Epoch 11: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 685s 12s/step - accuracy: 0.2747 - loss: 2.5478 - val_accuracy: 0.2769 - val_loss: 2.6677 - learning_rate: 1.0000e-04
Epoch 12/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3095 - loss: 2.4591
Epoch 12: val_accuracy did not improve from 0.27692
57/57 ━━━━━━━━━━━━━━━━━━━━ 654s 12s/step - accuracy: 0.3033 - loss: 2.4706 - val_accuracy: 0.2538 - val_loss: 2.6263 - learning_rate: 1.0000e-04
Epoch 13/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3516 - loss: 2.3817
Epoch 13: val_accuracy improved from 0.27692 to 0.30256, saving model to vgg16_best_model.h5



Epoch 13: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 710s 12s/step - accuracy: 0.3319 - loss: 2.4042 - val_accuracy: 0.3026 - val_loss: 2.5983 - learning_rate: 1.0000e-04
Epoch 14/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3435 - loss: 2.3343
Epoch 14: val_accuracy did not improve from 0.30256
57/57 ━━━━━━━━━━━━━━━━━━━━ 652s 11s/step - accuracy: 0.3352 - loss: 2.3681 - val_accuracy: 0.2897 - val_loss: 2.5833 - learning_rate: 1.0000e-04
Epoch 15/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3634 - loss: 2.2883
Epoch 15: val_accuracy did not improve from 0.30256
57/57 ━━━━━━━━━━━━━━━━━━━━ 684s 11s/step - accuracy: 0.3582 - loss: 2.3154 - val_accuracy: 0.2795 - val_loss: 2.5692 - learning_rate: 1.0000e-04
Epoch 16/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3880 - loss: 2.1550
Epoch 16: val_accuracy improved from 0.30256 to 0.30769, saving model to vgg16_best_model.h5



Epoch 16: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 684s 12s/step - accuracy: 0.3714 - loss: 2.2270 - val_accuracy: 0.3077 - val_loss: 2.5391 - learning_rate: 1.0000e-04
Epoch 17/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3847 - loss: 2.1560 
Epoch 17: val_accuracy did not improve from 0.30769
57/57 ━━━━━━━━━━━━━━━━━━━━ 657s 12s/step - accuracy: 0.3753 - loss: 2.1811 - val_accuracy: 0.2795 - val_loss: 2.5014 - learning_rate: 1.0000e-04
Epoch 18/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.4095 - loss: 2.1367
Epoch 18: val_accuracy did not improve from 0.30769
57/57 ━━━━━━━━━━━━━━━━━━━━ 709s 12s/step - accuracy: 0.3918 - loss: 2.1659 - val_accuracy: 0.2897 - val_loss: 2.4942 - learning_rate: 1.0000e-04
Epoch 19/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.4384 - loss: 2.0419
Epoch 19: val_accuracy did not improve from 0.30769
57/57 ━━━━━━━━━━━━━━━━━━━━ 655s 12s/step - accuracy: 0.4181 - loss: 2.0788 - val_accuracy: 0.2846 - 


Epoch 20: finished saving model to vgg16_best_model.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 654s 12s/step - accuracy: 0.4297 - loss: 2.0447 - val_accuracy: 0.3154 - val_loss: 2.4703 - learning_rate: 1.0000e-04


In [13]:
test_loss, test_acc = model.evaluate(test_ds)

print("Test Accuracy:", test_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 123s 9s/step - accuracy: 0.2410 - loss: 2.6768
Test Accuracy: 0.2410256415605545
